## 중요 내용
## 여러개의 데이터프레임을 결합하기
> DB는 각각의 엔티티별로 나누는걸 좋아함.  
> 스키마를 활용.  
> SQL에 있는건 판다스에 다 있음.  
> 데이터분석을 위해 여러 데이터셋을 가져와서 하나의 데이터프레임으로 병합하여 분석하는 것이 일반적임.  

데이터베이스 SQL의 테이블의 json 같은 기능

In [1]:
# 데이터는 보통 관측 단위별로 데이터를 표로 구성함.

#예1) 주식데이터 분석 : 기업정도 데이터셋 + 주식가격 데이터셋
#     ==> AI 관련 기업의 주가 정보 분석

#예2) 데이터의 크기가 커서 적당한 단위로 데이터 파일을 나누어 관리. (분기별 Q1, Q2, ..)

#예3) 시계열 데이털르 날짜별로 분리해서 관리. [주가, 기온, 판매량처럼 일정한 시간 간겨을 두고 순서
#     매주 빌보드 차트

### 기업정보 + 주식가격 데이터 결합
**"AI 기업 그룹의 주가 분석"**
> 서로 다른 구조와 정보를 가진 데이터셋을 '키(key)'로 연결해서 새로운 인사이트를 도출하는 것임

In [12]:
import pandas as pd

company_df= pd.read_csv('./data_set/데이터결합분리/company.csv')
stock_df= pd.read_csv('./data_set/데이터결합분리/stock.csv') # 기간 : 3일치..

In [13]:
#기업정보 확인
company_df

,기업코드,기업명,산업군
0,A001,알파테크,AI
1,A002,베타모터스,자동차
2,A003,감마AI,AI
3,A004,델타스틸,철강


In [14]:
#주가정보 확인
stock_df

,기업코드,기준일자,종가
0,A001,2025-01-01,120000
1,A001,2025-01-02,121500
2,A001,2025-01-03,119000
3,A002,2025-01-01,80000
4,A002,2025-01-02,80500
5,A002,2025-01-03,79000
6,A003,2025-01-01,150000
7,A003,2025-01-02,152000
8,A003,2025-01-03,151000
9,A004,2025-01-01,40000


In [11]:
# 서로 다른 csv 파일 두개를 '기업코드'라는 공통 열(column)을 기준으로 결합(병합 merge)해서 AI 기업의 주가만 분석하기!
merged_df= pd.merge(company_df, stock_df, on='기업코드')
merged_df

,기업코드,기업명,산업군,기준일자,종가
0,A001,알파테크,AI,2025-01-01,120000
1,A001,알파테크,AI,2025-01-02,121500
2,A001,알파테크,AI,2025-01-03,119000
3,A002,베타모터스,자동차,2025-01-01,80000
4,A002,베타모터스,자동차,2025-01-02,80500
5,A002,베타모터스,자동차,2025-01-03,79000
6,A003,감마AI,AI,2025-01-01,150000
7,A003,감마AI,AI,2025-01-02,152000
8,A003,감마AI,AI,2025-01-03,151000
9,A004,델타스틸,철강,2025-01-01,40000


In [16]:
# AI 산업군만 추출하기
ai_stock_df= merged_df[ merged_df['산업군']=='AI' ].copy() #인덱싱된 데이터프레임을 새로 생성
ai_stock_df

,기업코드,기업명,산업군,기준일자,종가
0,A001,알파테크,AI,2025-01-01,120000
1,A001,알파테크,AI,2025-01-02,121500
2,A001,알파테크,AI,2025-01-03,119000
6,A003,감마AI,AI,2025-01-01,150000
7,A003,감마AI,AI,2025-01-02,152000
8,A003,감마AI,AI,2025-01-03,151000


In [17]:
# 분석할때 필요한 변수들만 확인하기
ai_stock_df[['기업명','기준일자','종가']]

,기업명,기준일자,종가
0,알파테크,2025-01-01,120000
1,알파테크,2025-01-02,121500
2,알파테크,2025-01-03,119000
6,감마AI,2025-01-01,150000
7,감마AI,2025-01-02,152000
8,감마AI,2025-01-03,151000


In [18]:
# 수집 및 전처리단계가 마무리 되면.. 본격적으로 가설에 대한 분석을 시작!

# 분석질문
# "알파테크와 감마AI 중 단기적으로 더 안정적인 기업은 어디인가?"

# 수행내역
# 날짜에 따라 주가가 어떻게 변화하는가? --- 변동폭이 큰 기업은?
ai_stock_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 8
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   기업코드    6 non-null      object
 1   기업명     6 non-null      object
 2   산업군     6 non-null      object
 3   기준일자    6 non-null      object
 4   종가      6 non-null      int64 
dtypes: int64(1), object(4)
memory usage: 288.0+ bytes


In [19]:
#기준일자의 자료형이 object문자열 임. 그럼 날자에 관련된 여러기능 사용이 제한됨
#보통 날짜 자료형으로 변환
ai_stock_df['기준일자']= pd.to_datetime(ai_stock_df['기준일자'])
ai_stock_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6 entries, 0 to 8
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   기업코드    6 non-null      object        
 1   기업명     6 non-null      object        
 2   산업군     6 non-null      object        
 3   기준일자    6 non-null      datetime64[ns]
 4   종가      6 non-null      int64         
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 288.0+ bytes


In [20]:
# 날짜별 주가 변동값을 '일별변화'컬룸에 저장
ai_stock_df['일별변화']= ai_stock_df.groupby('기업명')['종가'].diff()
ai_stock_df

,기업코드,기업명,산업군,기준일자,종가,일별변화
0,A001,알파테크,AI,2025-01-01,120000,NaN
1,A001,알파테크,AI,2025-01-02,121500,1500.0
2,A001,알파테크,AI,2025-01-03,119000,-2500.0
6,A003,감마AI,AI,2025-01-01,150000,NaN
7,A003,감마AI,AI,2025-01-02,152000,2000.0
8,A003,감마AI,AI,2025-01-03,151000,-1000.0


In [22]:
# 변동폭 분석-- 누가 더 위험??

# 각 회사들의 평균증가 대비 얼마나 변동(차이:편차)이 심한지..확인..표준편차 std, 최대, 최소
aa= ai_stock_df.groupby('기업명')['종가'].describe() 
aa= ai_stock_df.groupby('기업명')['종가'].agg(['mean','std','min','max'])
aa # 알파테크의 변동폭이 더 크다는 것을 확인

,mean,std,min,max
기업명,,,,
감마AI,151000.000000,1000.000000,150000,152000
알파테크,120166.666667,1258.305739,119000,121500


In [24]:
# 만약, 단기 투자를 했다면.. 누가 더 수익률이 좋을까?
# 기업별 수익률 계산
수익율_df= ai_stock_df.groupby('기업명')[['종가']].apply(
    lambda x: (x.iloc[-1]-x.iloc[0])/x.iloc[0] * 100
)
수익율_df
#해석
#알파테크: 단기 하락
#감마AI : 단기 상승

# 상승과 하락의 원인을 분석

,종가
기업명,
감마AI,0.666667
알파테크,-0.833333


## 온라인 쇼핑몰 분기별 매출 데이터
 "어? 이 분기 매출이 왜 떨어졌지?" 같은 질문이 가능!!

In [25]:
# 온라인 쇼핑몰이 매출데이터를 분기별로 나누어 파일로 관리..
q1= pd.read_csv('./data_set/데이터결합분리/sales_2024_Q1.csv')
q2= pd.read_csv('./data_set/데이터결합분리/sales_2024_Q2.csv')
q3= pd.read_csv('./data_set/데이터결합분리/sales_2024_Q3.csv')
q4= pd.read_csv('./data_set/데이터결합분리/sales_2024_Q4.csv')

In [26]:
q1

,order_id,order_date,product,category,price,quantity,revenue
0,1001,2024-01-05,무선마우스,전자기기,25000,2,50000
1,1002,2024-02-10,기계식키보드,전자기기,85000,1,85000
2,1003,2024-03-15,텀블러,생활용품,18000,3,54000


In [27]:
q2

,order_id,order_date,product,category,price,quantity,revenue
0,2001,2024-04-02,노트북가방,패션,45000,1,45000
1,2002,2024-05-18,무선마우스,전자기기,25000,3,75000
2,2003,2024-06-27,책상스탠드,생활용품,32000,2,64000


In [28]:
q3

,order_id,order_date,product,category,price,quantity,revenue
0,3001,2024-07-08,기계식키보드,전자기기,85000,2,170000
1,3002,2024-08-14,블루투스스피커,전자기기,120000,1,120000
2,3003,2024-09-21,텀블러,생활용품,18000,4,72000


In [29]:
q4

,order_id,order_date,product,category,price,quantity,revenue
0,4001,2024-10-03,무선마우스,전자기기,25000,5,125000
1,4002,2024-11-11,겨울후드티,패션,62000,2,124000
2,4003,2024-12-24,책상스탠드,생활용품,32000,3,96000


In [30]:
#각 분기가 별도의 데이터셋이어서 분기별 차이들을 한눈에 확인 안됨.
#결합( merge???  횡결합X ... 위 아래 결합. 종결합... concat )
sales_2024= pd.concat([q1,q2,q3,q4])
sales_2024

,order_id,order_date,product,category,price,quantity,revenue
0,1001,2024-01-05,무선마우스,전자기기,25000,2,50000
1,1002,2024-02-10,기계식키보드,전자기기,85000,1,85000
2,1003,2024-03-15,텀블러,생활용품,18000,3,54000
0,2001,2024-04-02,노트북가방,패션,45000,1,45000
1,2002,2024-05-18,무선마우스,전자기기,25000,3,75000
2,2003,2024-06-27,책상스탠드,생활용품,32000,2,64000
0,3001,2024-07-08,기계식키보드,전자기기,85000,2,170000
1,3002,2024-08-14,블루투스스피커,전자기기,120000,1,120000
2,3003,2024-09-21,텀블러,생활용품,18000,4,72000
0,4001,2024-10-03,무선마우스,전자기기,25000,5,125000


In [33]:
#데이터셋의 구조 확인
sales_2024.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12 entries, 0 to 2
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    12 non-null     int64         
 1   order_date  12 non-null     object        
 2   product     12 non-null     object        
 3   category    12 non-null     object        
 4   price       12 non-null     int64         
 5   quantity    12 non-null     int64         
 6   revenue     12 non-null     int64         
 7   order date  12 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(4), object(3)
memory usage: 864.0+ bytes


In [34]:
#인덱스 번호 이상...하니..concat할때.. 기존 index를 무시하도록..
sales_2024= pd.concat([q1,q2,q3,q4], ignore_index=True)
sales_2024

,order_id,order_date,product,category,price,quantity,revenue
0,1001,2024-01-05,무선마우스,전자기기,25000,2,50000
1,1002,2024-02-10,기계식키보드,전자기기,85000,1,85000
2,1003,2024-03-15,텀블러,생활용품,18000,3,54000
3,2001,2024-04-02,노트북가방,패션,45000,1,45000
4,2002,2024-05-18,무선마우스,전자기기,25000,3,75000
5,2003,2024-06-27,책상스탠드,생활용품,32000,2,64000
6,3001,2024-07-08,기계식키보드,전자기기,85000,2,170000
7,3002,2024-08-14,블루투스스피커,전자기기,120000,1,120000
8,3003,2024-09-21,텀블러,생활용품,18000,4,72000
9,4001,2024-10-03,무선마우스,전자기기,25000,5,125000


In [35]:
#기본 데이터의 구조와 기초 통계
sales_2024.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   order_id    12 non-null     int64 
 1   order_date  12 non-null     object
 2   product     12 non-null     object
 3   category    12 non-null     object
 4   price       12 non-null     int64 
 5   quantity    12 non-null     int64 
 6   revenue     12 non-null     int64 
dtypes: int64(4), object(3)
memory usage: 804.0+ bytes


In [36]:
sales_2024.describe()

,order_id,price,quantity,revenue
count,12.000000,12.000000,12.000000,12.000000
mean,2502.000000,47666.666667,2.416667,90000.000000
std,1167.748728,33062.841910,1.240112,38004.784388
min,1001.000000,18000.000000,1.000000,45000.000000
25%,1751.500000,25000.000000,1.750000,61500.000000
50%,2502.000000,32000.000000,2.000000,80000.000000
75%,3252.500000,67750.000000,3.000000,121000.000000
max,4003.000000,120000.000000,5.000000,170000.000000


In [37]:
# 데이터 분석 실습
#1. 분기별 매출 비교 -- 날짜계산이 필요.. [주문일자]컬룸의 자료형은 datetime으로 변환
sales_2024['order_date']= pd.to_datetime(sales_2024['order_date'])

#datetime으로 바꾸면 좋은점.. 특정 기간별(일/월/주/분기/년)로 집계를 해주는 기능 .dt.to_priod()
#날짜를 2024Q1
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('Q') #분기 단위
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('M') #month 단위
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('Y') #year 단위
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('D') #day 단위(소문자도 OK)
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('h') #hour 단위(소문자)
sales_2024['quarter'] = sales_2024['order_date'].dt.to_period('Q') #분기 단위

sales_2024.groupby('quarter')['revenue'].sum()
#결과를 보고 해석 : 3분기에 매출이 2배 뛰었네..!!!  3분기에 뭔가 이벤트가 있었나? 프로모션?. 신제품?

quarter
2024Q1    189000
2024Q2    184000
2024Q3    362000
2024Q4    345000
Freq: Q-DEC, Name: revenue, dtype: int64

In [38]:
#2. 카테고리별 매출 분석
sales_2024.groupby('category')['revenue'].sum().sort_values(ascending=False)

category
전자기기    625000
생활용품    286000
패션      169000
Name: revenue, dtype: int64

In [39]:
#3. 가장 많이 팔린 제품 TOP3
sales_2024.groupby('product')['quantity'].sum().sort_values(ascending=False).head(3)

product
무선마우스    10
텀블러       7
책상스탠드     5
Name: quantity, dtype: int64

In [41]:
# concat 할때 컬룸이 다르면???

#예) 프로모션 이벤트 발생(Q3부터 제품별 할인 컬룸 도입)

#Q3, Q4에 discount 컬룸 추가 ( 제품별 할인율을 저장 )
discount_map= {
    '무선마우스':0.10,
    '기계식키보드':0.15,
    '블루투스스피커':0.20,
    '책상스탠드':0.00,
    '겨울후드티':0.15
}

# q3 데이터프레임에서 product 제품명에 까라 위 dict의 해당하는 할인율을 ['discount']칸에 저장
q3['discount']= q3['product'].map(discount_map).fillna(0) #해당 제품명을 dict의 키로 매핑하여 값을 추출
q4['discount']= q4['product'].map(discount_map).fillna(0)

In [42]:
q3

,order_id,order_date,product,category,price,quantity,revenue,discount
0,3001,2024-07-08,기계식키보드,전자기기,85000,2,170000,0.15
1,3002,2024-08-14,블루투스스피커,전자기기,120000,1,120000,0.20
2,3003,2024-09-21,텀블러,생활용품,18000,4,72000,0.00


In [43]:
q4

,order_id,order_date,product,category,price,quantity,revenue,discount
0,4001,2024-10-03,무선마우스,전자기기,25000,5,125000,0.10
1,4002,2024-11-11,겨울후드티,패션,62000,2,124000,0.15
2,4003,2024-12-24,책상스탠드,생활용품,32000,3,96000,0.00


In [44]:
# q1,q2,q3,q4 4분기 데이터를 결합(concat)
sales_2024= pd.concat([q1,q2,q3,q4], ignore_index=True)
sales_2024

,order_id,order_date,product,category,price,quantity,revenue,discount
0,1001,2024-01-05,무선마우스,전자기기,25000,2,50000,NaN
1,1002,2024-02-10,기계식키보드,전자기기,85000,1,85000,NaN
2,1003,2024-03-15,텀블러,생활용품,18000,3,54000,NaN
3,2001,2024-04-02,노트북가방,패션,45000,1,45000,NaN
4,2002,2024-05-18,무선마우스,전자기기,25000,3,75000,NaN
5,2003,2024-06-27,책상스탠드,생활용품,32000,2,64000,NaN
6,3001,2024-07-08,기계식키보드,전자기기,85000,2,170000,0.15
7,3002,2024-08-14,블루투스스피커,전자기기,120000,1,120000,0.20
8,3003,2024-09-21,텀블러,생활용품,18000,4,72000,0.00
9,4001,2024-10-03,무선마우스,전자기기,25000,5,125000,0.10


In [45]:
# 결측치를 기반으로 정책 변경 시점을 데이터로 자연스럽게 드러남.
# 실무에서 새로운 컬룸이 추가되는 일은 꽤 빈번함.

### merge 할때 on에 대응하는 행 개수가 다를때 사용하는 how속성
how속성값 : left, right, outer, inner 에 대해 알아보기

In [46]:
#어떤 쇼핑몰의 고객데이터와 주문데이터의 결합
customers= pd.read_csv('./data_set/데이터결합분리/customers.csv')
orders= pd.read_csv('./data_set/데이터결합분리/orders.csv')

In [47]:
#고개정보 확인
customers

,customer_id,customer_name,region
0,C001,김민수,서울
1,C002,이영희,부산
2,C005,박철수,대전


In [48]:
#주문정보 확인
orders

,order_id,customer_id,product,amount
0,1001,C001,무선마우스,50000
1,1002,C002,기계식키보드,85000
2,1003,C003,텀블러,54000
3,1004,C004,책상스탠드,64000


In [49]:
# C003, C004 -> 주문은 있지만 고객 정보 없음 [회원 탈퇴. 비회원 주문 ]
# C005       -> 고객정보는 있는데 주문이 없는 경우 

In [50]:
# merge 실습 
# 공통 key : 'customer_id'  #양쪽 데이터셋에 모두 있으면서 관측값을 구별할 수 있는 식별값

# 양쪽 데이터의 key값의 정보균형이 다를때.. 어떻게 할지를 결정하는 설정값 how
#1) inner (기본값) -- 교집합 (양쪽 모두에 있는 데이터만 병합되어 만들어짐)
pd.merge(orders, customers, on='customer_id', how='inner')

,order_id,customer_id,product,amount,customer_name,region
0,1001,C001,무선마우스,50000,김민수,서울
1,1002,C002,기계식키보드,85000,이영희,부산


In [51]:
#2) left (실무에서 많이 사용) -- 왼쪽에 있는 데이터셋들만 병합되어 만들어짐(값이 없으면 결측값)
pd.merge(orders, customers, on='customer_id', how='left')

,order_id,customer_id,product,amount,customer_name,region
0,1001,C001,무선마우스,50000,김민수,서울
1,1002,C002,기계식키보드,85000,이영희,부산
2,1003,C003,텀블러,54000,NaN,NaN
3,1004,C004,책상스탠드,64000,NaN,NaN


In [52]:
#2) right -- 오른쪽에 있는 데이터셋들만 병합되어 만들어짐(값이 없으면 결측값) --고객관점에서 분석
pd.merge(orders, customers, on='customer_id', how='right')

,order_id,customer_id,product,amount,customer_name,region
0,1001.0,C001,무선마우스,50000.0,김민수,서울
1,1002.0,C002,기계식키보드,85000.0,이영희,부산
2,NaN,C005,NaN,NaN,박철수,대전


In [53]:
#4) outer -- 양쪽에 있는 모든 데이터셋들 병합되어 만들어짐(값이 없으면 결측값) -- 합집합
pd.merge(orders, customers, on='customer_id', how='outer')

,order_id,customer_id,product,amount,customer_name,region
0,1001.0,C001,무선마우스,50000.0,김민수,서울
1,1002.0,C002,기계식키보드,85000.0,이영희,부산
2,1003.0,C003,텀블러,54000.0,NaN,NaN
3,1004.0,C004,책상스탠드,64000.0,NaN,NaN
4,NaN,C005,NaN,NaN,박철수,대전


### 여러개의 파일들 한방에 불러오기
- 예) 주식정보를 매일 저장한다면 날짜별로 파일들이 존재함.
- 예) 매주 빌보드 차트를 파일일별로 관리.

In [54]:
#빌보드 차트의 데이터셋이 너무 많음... 76주간의 데이터파일..

#파일명의 일부만 다를 경우 한방에 불러올 수 있음
from pathlib import Path

billboard_data_files= Path(".").glob("./data/billboard_by_week/billboard-*.csv")

billboard_data_files= sorted( list(billboard_data_files) )
billboard_data_files

[WindowsPath('data/billboard_by_week/billboard-01.csv'),
 WindowsPath('data/billboard_by_week/billboard-02.csv'),
 WindowsPath('data/billboard_by_week/billboard-03.csv'),
 WindowsPath('data/billboard_by_week/billboard-04.csv'),
 WindowsPath('data/billboard_by_week/billboard-05.csv'),
 WindowsPath('data/billboard_by_week/billboard-06.csv'),
 WindowsPath('data/billboard_by_week/billboard-07.csv'),
 WindowsPath('data/billboard_by_week/billboard-08.csv'),
 WindowsPath('data/billboard_by_week/billboard-09.csv'),
 WindowsPath('data/billboard_by_week/billboard-10.csv'),
 WindowsPath('data/billboard_by_week/billboard-11.csv'),
 WindowsPath('data/billboard_by_week/billboard-12.csv'),
 WindowsPath('data/billboard_by_week/billboard-13.csv'),
 WindowsPath('data/billboard_by_week/billboard-14.csv'),
 WindowsPath('data/billboard_by_week/billboard-15.csv'),
 WindowsPath('data/billboard_by_week/billboard-16.csv'),
 WindowsPath('data/billboard_by_week/billboard-17.csv'),
 WindowsPath('data/billboard_by

In [55]:
billboard_df_01= pd.read_csv(billboard_data_files[0])
billboard_df_02= pd.read_csv(billboard_data_files[1])
billboard_df_03= pd.read_csv(billboard_data_files[2])
#반복문으로 데이터프레임 모두 가져올 수 있음.

In [56]:
billboard_df_01

,year,artist,track,time,date.entered,week,rating
0,2000,2 Pac,Baby Don't Cry (Keep...,4:22,2000-02-26,wk1,87.0
1,2000,2Ge+her,The Hardest Part Of ...,3:15,2000-09-02,wk1,91.0
2,2000,3 Doors Down,Kryptonite,3:53,2000-04-08,wk1,81.0
3,2000,3 Doors Down,Loser,4:24,2000-10-21,wk1,76.0
4,2000,504 Boyz,Wobble Wobble,3:35,2000-04-15,wk1,57.0
...,...,...,...,...,...,...,...
312,2000,Yankee Grey,Another Nine Minutes,3:10,2000-04-29,wk1,86.0
313,2000,"Yearwood, Trisha",Real Live Woman,3:55,2000-04-01,wk1,85.0
314,2000,Ying Yang Twins,Whistle While You Tw...,4:19,2000-03-18,wk1,95.0
315,2000,Zombie Nation,Kernkraft 400,3:30,2000-09-02,wk1,99.0


In [57]:
billboard_df_02

,year,artist,track,time,date.entered,week,rating
0,2000,2 Pac,Baby Don't Cry (Keep...,4:22,2000-02-26,wk2,82.0
1,2000,2Ge+her,The Hardest Part Of ...,3:15,2000-09-02,wk2,87.0
2,2000,3 Doors Down,Kryptonite,3:53,2000-04-08,wk2,70.0
3,2000,3 Doors Down,Loser,4:24,2000-10-21,wk2,76.0
4,2000,504 Boyz,Wobble Wobble,3:35,2000-04-15,wk2,34.0
...,...,...,...,...,...,...,...
312,2000,Yankee Grey,Another Nine Minutes,3:10,2000-04-29,wk2,83.0
313,2000,"Yearwood, Trisha",Real Live Woman,3:55,2000-04-01,wk2,83.0
314,2000,Ying Yang Twins,Whistle While You Tw...,4:19,2000-03-18,wk2,94.0
315,2000,Zombie Nation,Kernkraft 400,3:30,2000-09-02,wk2,99.0


In [58]:
# 이 데이터프레임을 결합하여 분석 수행!!!